## 1 · Setup

In [ ]:
import importlib, subprocess, sys

REQUIRED = {
    "sklearn":       "scikit-learn",
    "pandas":        "pandas",
    "numpy":         "numpy",
    "matplotlib":    "matplotlib",
    "seaborn":       "seaborn",
    "gensim":        "gensim",
    "emoji":         "emoji",
    "tqdm":          "tqdm",
    "pyarrow":       "pyarrow",
    "nltk":          "nltk",
    "tensorflow":    "tensorflow",
}

missing = []
for mod, spec in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(spec)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies OK.")

## 2 · Imports

In [ ]:
import os, re, json, math, random, gc, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)
from sklearn.svm import SVC

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense, Dropout, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["figure.dpi"] = 110

print("TensorFlow:", tf.__version__)

## 3 · Configuration block (all hyperparameters live here)

In [ ]:
# Auto-detect environment (Colab vs local)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    PROJECT_ROOT = Path("/content")
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT, "| In Colab:", IN_COLAB)

@dataclass
class CFG:
    # paths (all relative to PROJECT_ROOT)
    csv_name: str    = "political_reddit_100k.csv"
    splits_dir: str  = "splits"
    models_dir: str  = "models"
    results_dir: str = "results"
    figures_dir: str = "figures"
    tables_dir: str  = "tables"
    cache_dir: str   = "cache"

    # data
    text_col: str  = "text"
    label_col: str = "label"
    seed_split: int = 42

    max_seq_len: int = 128

cfg = CFG()

# Resolve CSV path (Colab Files tab or local cwd)
csv_candidates = [
    PROJECT_ROOT / cfg.csv_name,
    Path.cwd() / cfg.csv_name,
    Path("/content") / cfg.csv_name,
]
CSV_PATH = next((p for p in csv_candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        f"Could not find '{cfg.csv_name}'. Looked in:\n  "
        + "\n  ".join(str(p) for p in csv_candidates)
        + "\nIn Colab: drag the file into the Files panel (it lands in /content/)."
    )
print("CSV path:", CSV_PATH)

# Create output dirs
ROOT = PROJECT_ROOT
for d in [cfg.splits_dir, cfg.models_dir, cfg.results_dir,
          cfg.figures_dir, cfg.tables_dir, cfg.cache_dir]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))

## 4 · Reproducibility + mixed-precision auto-detect

In [ ]:
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(cfg.seed_split)

## 5 · Data loading

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape)
df.head(3)


## 6 · Integrity checks

In [ ]:
assert cfg.text_col in df.columns and cfg.label_col in df.columns, "Missing columns."
df[cfg.label_col] = df[cfg.label_col].astype(int)
df[cfg.text_col]  = df[cfg.text_col].astype(str)

n_null  = df[cfg.text_col].isna().sum()
n_empty = (df[cfg.text_col].str.strip() == "").sum()
n_dup   = df.duplicated(subset=[cfg.text_col]).sum()
print(f"rows={len(df)}  nulls={n_null}  empty={n_empty}  dup_text={n_dup}")
print("\nLabel counts:\n", df[cfg.label_col].value_counts().sort_index())
assert set(df[cfg.label_col].unique()) == {0, 1}, "Labels must be {0, 1}"


## 7 · Compact EDA

In [ ]:
df["_char_len"] = df[cfg.text_col].str.len()
df["_word_len"] = df[cfg.text_col].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
sns.countplot(x=cfg.label_col, data=df, ax=axes[0])
axes[0].set_title("Class balance")
axes[0].set_xticklabels(["Democrat (0)", "Republican (1)"])

clip = df["_char_len"].quantile(0.99)
sns.histplot(df["_char_len"].clip(upper=clip), bins=60, ax=axes[1])
axes[1].set_title(f"Char length (clipped p99={int(clip)})")

sns.histplot(df["_word_len"].clip(upper=df["_word_len"].quantile(0.99)),
             bins=60, ax=axes[2])
axes[2].set_title("Word length (clipped p99)")
plt.tight_layout()
plt.savefig(ROOT / cfg.figures_dir / "eda_overview.png", bbox_inches="tight")
plt.show()

summary = pd.DataFrame({
    "n":              df.groupby(cfg.label_col).size(),
    "char_len_p50":   df.groupby(cfg.label_col)["_char_len"].median().round(1),
    "char_len_p95":   df.groupby(cfg.label_col)["_char_len"].quantile(0.95).round(1),
    "word_len_p50":   df.groupby(cfg.label_col)["_word_len"].median().round(1),
    "word_len_p95":   df.groupby(cfg.label_col)["_word_len"].quantile(0.95).round(1),
})
display(summary)
df = df.drop(columns=["_char_len", "_word_len"])


## 8 · Preprocessing (anti-leakage)

Scrubs subreddit name leaks, mod-bot residue, URLs, user mentions, markdown noise. Demojizes emoji.


In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text: str) -> str:
    tokens = nltk.word_tokenize(str(text))
    
    processed_tokens = [
        stemmer.stem(t)
        for t in tokens
        if t.lower() not in stop_words
    ]
    
    return " ".join(processed_tokens)

# Apply + cache
cache_path = ROOT / cfg.cache_dir / "preprocessed.parquet"
if cache_path.exists():
    df_pp = pd.read_parquet(cache_path)
    print("Loaded cached preprocessed:", df_pp.shape)
else:
    tqdm.pandas(desc="preprocess")
    df_pp = df.copy()
    df_pp["text"] = df_pp[cfg.text_col].progress_apply(preprocess)
    keep = df_pp["text"].str.len() > 0
    print(f"Dropping {(~keep).sum()} rows that became empty after preprocessing")
    df_pp = df_pp[keep].reset_index(drop=True)[["text", cfg.label_col]]
    df_pp.to_parquet(cache_path, index=False)

print(df_pp.head(3))

## 9 · Exact 80 / 20 train / test split

In [ ]:
def split_data(df_in: pd.DataFrame, seed: int):
    train_df, test_df = train_test_split(
        df_in, test_size=0.20, random_state=seed)
    return (train_df.reset_index(drop=True),
            test_df.reset_index(drop=True))

splits_dir = ROOT / cfg.splits_dir
if not (splits_dir / "train.csv").exists():
    tr, te = split_data(df_pp, cfg.seed_split)
    tr.to_csv(splits_dir / "train.csv", index=False)
    te.to_csv(splits_dir / "test.csv",  index=False)

train_df = pd.read_csv(splits_dir / "train.csv")
test_df  = pd.read_csv(splits_dir / "test.csv")

print("train:", train_df.shape, "test:", test_df.shape)
for name, d in [("train", train_df), ("test", test_df)]:
    print(name, "balance:", dict(d[cfg.label_col].value_counts().sort_index()))

## 10 · Token-length profiling

In [ ]:
sample = train_df["text"].sample(min(10000, len(train_df)), random_state=0).astype(str).tolist()
lens = np.array([len(x.split()) for x in sample])
print(f"word len  mean={lens.mean():.1f}  median={np.median(lens):.0f}  "
      f"p95={np.quantile(lens, 0.95):.0f}  p99={np.quantile(lens, 0.99):.0f}  "
      f"max={lens.max()}")

plt.figure(figsize=(7, 3))
plt.hist(np.clip(lens, 0, 512), bins=60)
plt.axvline(cfg.max_seq_len, color="red", linestyle="--", label=f"max_seq_len={cfg.max_seq_len}")
plt.title("Word length distribution (train sample)")
plt.xlabel("words"); plt.ylabel("count"); plt.legend()
plt.tight_layout()
plt.savefig(ROOT / cfg.figures_dir / "token_length.png", bbox_inches="tight")
plt.show()

print(f"Truncation rate at max_seq_len={cfg.max_seq_len}: {(lens > cfg.max_seq_len).mean()*100:.2f}%")

## 11 · Word2Vec Training (Library Defaults)

In [ ]:
train_toks = [str(t).split() for t in train_df["text"].tolist()]
test_toks  = [str(t).split() for t in test_df["text"].tolist()]

# CBOW (sg=0)
w2v_cbow_path = ROOT / cfg.cache_dir / "w2v_cbow.model"
if w2v_cbow_path.exists():
    w2v_cbow = Word2Vec.load(str(w2v_cbow_path))
else:
    w2v_cbow = Word2Vec(sentences=train_toks, sg=0)
    w2v_cbow.save(str(w2v_cbow_path))

# Skip-gram (sg=1)
w2v_sg_path = ROOT / cfg.cache_dir / "w2v_sg.model"
if w2v_sg_path.exists():
    w2v_sg = Word2Vec.load(str(w2v_sg_path))
else:
    w2v_sg = Word2Vec(sentences=train_toks, sg=1)
    w2v_sg.save(str(w2v_sg_path))

print("CBOW vocab size:", len(w2v_cbow.wv))
print("Skip-gram vocab size:", len(w2v_sg.wv))

def get_mean_embeddings(tokens_list, model):
    X = []
    vector_size = model.vector_size
    for tokens in tokens_list:
        vecs = [model.wv[t] for t in tokens if t in model.wv]
        if len(vecs) > 0:
            X.append(np.mean(vecs, axis=0))
        else:
            X.append(np.zeros(vector_size))
    return np.array(X)

X_train_cbow_mean = get_mean_embeddings(train_toks, w2v_cbow)
X_test_cbow_mean = get_mean_embeddings(test_toks, w2v_cbow)

X_train_sg_mean = get_mean_embeddings(train_toks, w2v_sg)
X_test_sg_mean = get_mean_embeddings(test_toks, w2v_sg)

y_train = train_df[cfg.label_col].values
y_test = test_df[cfg.label_col].values

## 12 · SVM Baseline (CBOW and Skip-gram)

In [ ]:
# SVM with mean word embeddings (CBOW)
svm_cbow = SVC(kernel='linear', C=0.1)
svm_cbow.fit(X_train_cbow_mean, y_train)
svm_cbow_preds = svm_cbow.predict(X_test_cbow_mean)
print("SVM (CBOW) Test Accuracy:", accuracy_score(y_test, svm_cbow_preds))
print("SVM (CBOW) Classification Report:\n", classification_report(y_test, svm_cbow_preds))
print("SVM (CBOW) Confusion Matrix:\n", confusion_matrix(y_test, svm_cbow_preds))

# SVM with mean word embeddings (Skip-gram)
svm_sg = SVC(kernel='linear', C=0.1)
svm_sg.fit(X_train_sg_mean, y_train)
svm_sg_preds = svm_sg.predict(X_test_sg_mean)
print("\nSVM (Skip-gram) Test Accuracy:", accuracy_score(y_test, svm_sg_preds))
print("SVM (Skip-gram) Classification Report:\n", classification_report(y_test, svm_sg_preds))
print("SVM (Skip-gram) Confusion Matrix:\n", confusion_matrix(y_test, svm_sg_preds))

## 13 · CNN Baseline (CBOW and Skip-gram)

In [ ]:
def build_cnn(vector_size):
    model = Sequential([
        Input(shape=(vector_size, 1)),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# CNN CBOW
X_train_cbow_cnn = X_train_cbow_mean.reshape(-1, w2v_cbow.vector_size, 1)
X_test_cbow_cnn = X_test_cbow_mean.reshape(-1, w2v_cbow.vector_size, 1)

cnn_cbow = build_cnn(w2v_cbow.vector_size)
print("Training CNN (CBOW)...")
cnn_cbow.fit(X_train_cbow_cnn, y_train, batch_size=64, epochs=30, verbose=0)
cnn_cbow_probs = cnn_cbow.predict(X_test_cbow_cnn).flatten()
cnn_cbow_preds = (cnn_cbow_probs > 0.5).astype(int)
print("CNN (CBOW) Test Accuracy:", accuracy_score(y_test, cnn_cbow_preds))
print("CNN (CBOW) Classification Report:\n", classification_report(y_test, cnn_cbow_preds))

# CNN Skip-gram
X_train_sg_cnn = X_train_sg_mean.reshape(-1, w2v_sg.vector_size, 1)
X_test_sg_cnn = X_test_sg_mean.reshape(-1, w2v_sg.vector_size, 1)

cnn_sg = build_cnn(w2v_sg.vector_size)
print("\nTraining CNN (Skip-gram)...")
cnn_sg.fit(X_train_sg_cnn, y_train, batch_size=64, epochs=30, verbose=0)
cnn_sg_probs = cnn_sg.predict(X_test_sg_cnn).flatten()
cnn_sg_preds = (cnn_sg_probs > 0.5).astype(int)
print("CNN (Skip-gram) Test Accuracy:", accuracy_score(y_test, cnn_sg_preds))
print("CNN (Skip-gram) Classification Report:\n", classification_report(y_test, cnn_sg_preds))

## 14 · Data Preparation for Sequence Models

In [ ]:
# Prepare sequential word embeddings for LSTM and CNN-LSTM
train_texts = [" ".join(tokens) for tokens in train_toks]
test_texts  = [" ".join(tokens) for tokens in test_toks]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_texts)

vocab_size = len(tokenizer.word_index) + 1
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=cfg.max_seq_len, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=cfg.max_seq_len, padding='post')

emb_matrix_cbow = np.zeros((vocab_size, w2v_cbow.vector_size))
emb_matrix_sg = np.zeros((vocab_size, w2v_sg.vector_size))

for word, i in tokenizer.word_index.items():
    if word in w2v_cbow.wv:
        emb_matrix_cbow[i] = w2v_cbow.wv[word]
    if word in w2v_sg.wv:
        emb_matrix_sg[i] = w2v_sg.wv[word]

## 15 · LSTM Baseline (CBOW and Skip-gram)

In [ ]:
def build_lstm(emb_matrix, vector_size):
    model = Sequential([
        Embedding(vocab_size, vector_size, weights=[emb_matrix], input_length=cfg.max_seq_len, trainable=False),
        LSTM(128, dropout=0.4, recurrent_dropout=0.4), # paper mentions input + recurrent dropout
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# LSTM CBOW
lstm_cbow = build_lstm(emb_matrix_cbow, w2v_cbow.vector_size)
print("Training LSTM (CBOW)...")
lstm_cbow.fit(X_train_seq, y_train, batch_size=64, epochs=30, verbose=0)
lstm_cbow_probs = lstm_cbow.predict(X_test_seq).flatten()
lstm_cbow_preds = (lstm_cbow_probs > 0.5).astype(int)
print("LSTM (CBOW) Test Accuracy:", accuracy_score(y_test, lstm_cbow_preds))
print("LSTM (CBOW) Classification Report:\n", classification_report(y_test, lstm_cbow_preds))

# LSTM Skip-gram
lstm_sg = build_lstm(emb_matrix_sg, w2v_sg.vector_size)
print("\nTraining LSTM (Skip-gram)...")
lstm_sg.fit(X_train_seq, y_train, batch_size=64, epochs=30, verbose=0)
lstm_sg_probs = lstm_sg.predict(X_test_seq).flatten()
lstm_sg_preds = (lstm_sg_probs > 0.5).astype(int)
print("LSTM (Skip-gram) Test Accuracy:", accuracy_score(y_test, lstm_sg_preds))
print("LSTM (Skip-gram) Classification Report:\n", classification_report(y_test, lstm_sg_preds))

## 16 · CNN-LSTM Baseline (CBOW and Skip-gram)

In [ ]:
def build_cnn_lstm(emb_matrix, vector_size):
    model = Sequential([
        Embedding(vocab_size, vector_size, weights=[emb_matrix], input_length=cfg.max_seq_len, trainable=False),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        LSTM(128, dropout=0.4, recurrent_dropout=0.4),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# CNN-LSTM CBOW
cnn_lstm_cbow = build_cnn_lstm(emb_matrix_cbow, w2v_cbow.vector_size)
print("Training CNN-LSTM (CBOW)...")
cnn_lstm_cbow.fit(X_train_seq, y_train, batch_size=64, epochs=30, verbose=0)
cnn_lstm_cbow_probs = cnn_lstm_cbow.predict(X_test_seq).flatten()
cnn_lstm_cbow_preds = (cnn_lstm_cbow_probs > 0.5).astype(int)
print("CNN-LSTM (CBOW) Test Accuracy:", accuracy_score(y_test, cnn_lstm_cbow_preds))
print("CNN-LSTM (CBOW) Classification Report:\n", classification_report(y_test, cnn_lstm_cbow_preds))

# CNN-LSTM Skip-gram
cnn_lstm_sg = build_cnn_lstm(emb_matrix_sg, w2v_sg.vector_size)
print("\nTraining CNN-LSTM (Skip-gram)...")
cnn_lstm_sg.fit(X_train_seq, y_train, batch_size=64, epochs=30, verbose=0)
cnn_lstm_sg_probs = cnn_lstm_sg.predict(X_test_seq).flatten()
cnn_lstm_sg_preds = (cnn_lstm_sg_probs > 0.5).astype(int)
print("CNN-LSTM (Skip-gram) Test Accuracy:", accuracy_score(y_test, cnn_lstm_sg_preds))
print("CNN-LSTM (Skip-gram) Classification Report:\n", classification_report(y_test, cnn_lstm_sg_preds))